# E4 · So sánh mô hình · Chủ: Người 1 · Phục vụ Chương 5 (mục 5.4, RQ3)
LR và Naive Bayes trên Spark (cùng train/val/test), ghi cả thời gian và phần cứng → **T5.6**.
RoBERTa (GPU): xem TODO bên dưới. Nếu Người 1 quá tải, phần RoBERTa có thể giao cho Người 2 trong notebook riêng `E4b_roberta.ipynb`.

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "ten-cua-ban"     # điền tên để log kết quả
REPO  = "https://github.com/<ten-tai-khoan>/ai-text-detection.git"   # sửa thành repo của nhóm

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

In [ ]:
from aitext import data, models, evaluate
train, val, test = data.load_splits(spark)
train = data.with_class_weight(train).cache(); rows = []
for clf in ["lr", "nb"]:
    t0 = time.time(); m = models.fit(train, defended=True, clf=clf); dt_ = time.time() - t0
    thr = evaluate.thr_at_fpr(evaluate.score(m, val, models.text_col(True)))
    r = evaluate.metrics(evaluate.score(m, test, models.text_col(True)), thr)
    evaluate.log_run("E4", OWNER, f"{clf}|clean", r, notes=f"train {dt_:.0f}s, Colab CPU")
    rows.append({"mô hình": clf, "train (s)": dt_, "phần cứng": "Colab CPU", **r})
t56 = pd.DataFrame(rows); display(t56.round(4)); evaluate.save_table(t56, "T5.6_model_compare")

## TODO: RoBERTa trên tập con (runtime GPU)
1. Lấy khoảng 20.000–50.000 dòng cân bằng từ `clean_train` và toàn bộ `clean_val`, `clean_test` → pandas.
2. Fine-tune `roberta-base` (1 epoch, max_length 256) bằng `transformers`.
3. Chọn ngưỡng trên val (FPR 5%), đo trên test bằng cùng định nghĩa TPR/FPR; ghi thời gian + loại GPU.
4. Thêm 1 dòng vào T5.6. Mốc: chưa xong trước 04/11 thì bỏ, chỉ báo cáo LR và NB.